<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L10_Text_Generation_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L10: Text Generation Basics - Decoding Strategies and Sampling

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 10 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Understand different text generation strategies
- Master greedy search and its limitations
- Learn beam search for better quality generation
- Explore sampling strategies for diverse outputs
- Control generation with temperature parameter
- Apply top-k and top-p (nucleus) sampling
- Compare generation quality across strategies

---

## Concept: How Language Models Generate Text

**Text generation** is the process of producing coherent text sequences token by token.

### The Generation Process:

**At each step:**
1. Model receives input tokens
2. Predicts probability distribution over vocabulary
3. Selects next token based on strategy
4. Adds token to sequence
5. Repeats until stopping condition

```
Input: "The cat sat on the"
  |
  v
Model predicts: {"mat": 0.4, "floor": 0.3, "chair": 0.2, ...}
  |
  v
Strategy selects: "mat"
  |
  v
New input: "The cat sat on the mat"
```

### The Challenge:

**How do we choose the next token?**
- **Most likely token?** (Greedy) - Safe but repetitive
- **Best sequence?** (Beam Search) - Better quality but expensive
- **Random sampling?** (Sampling) - Diverse but risky

### Key Strategies:

1. **Greedy Search** - Always pick highest probability token
2. **Beam Search** - Keep top-k sequences, pick best overall
3. **Sampling** - Randomly sample from probability distribution
4. **Temperature** - Control randomness in sampling
5. **Top-k Sampling** - Sample from top-k most likely tokens
6. **Top-p (Nucleus) Sampling** - Sample from smallest set with cumulative probability p

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install transformers torch accelerate -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Step 2: Load Model and Tokenizer

print("Loading GPT-2 Model\n")
print("=" * 60)

# Load GPT-2 for text generation
model_name = "gpt2"
print(f"Model: {model_name}\n")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print()

# Test prompt
prompt = "Artificial intelligence is"
print(f"Test prompt: '{prompt}'")
print("\nWe'll use this prompt to compare different generation strategies.")

## Part 1: Greedy Search - The Simplest Strategy

**Greedy Search** always selects the token with the highest probability at each step.

### How It Works:

```
At each step:
  1. Get probability distribution
  2. Select argmax(probabilities)
  3. Add to sequence
```

### Advantages:
- Fast and simple
- Deterministic (same input = same output)
- Low computational cost

### Disadvantages:
- Can be repetitive
- May miss better overall sequences
- No diversity in outputs
- Prone to getting stuck in loops

---

In [ ]:
# Step 3: Greedy Search Generation

print("Greedy Search Generation\n")
print("=" * 60)

prompt = "Artificial intelligence is"
print(f"Prompt: {prompt}\n")

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt")

# Generate with greedy search (default)
print("Generating with greedy search...\n")
with torch.no_grad():
    outputs = model.generate(
        inputs['input_ids'],
        max_length=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False  # Greedy search
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Generated text:\n{generated_text}")
print()

# Generate multiple times to show determinism
print("Running greedy search 3 times to show determinism:\n")
for i in range(3):
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=50,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{i+1}. {text}")
    print()

print("Notice: All outputs are identical (deterministic)")
print("\n" + "=" * 60)

## Part 2: Beam Search - Finding Better Sequences

**Beam Search** maintains multiple hypotheses (beams) and selects the sequence with highest overall probability.

### How It Works:

```
At each step:
  1. For each beam, get top-k next tokens
  2. Score all beam * k combinations
  3. Keep top num_beams sequences
  4. Continue until all beams end or max_length
```

### Key Parameters:
- **num_beams**: Number of beams to keep (typically 4-10)
- **early_stopping**: Stop when num_beams complete sequences found

### Advantages:
- Better quality than greedy search
- Finds higher probability sequences
- Good for tasks requiring accuracy

### Disadvantages:
- More computationally expensive
- Still deterministic
- Can be generic or boring
- May produce repetitive text

---

In [ ]:
# Step 4: Beam Search Generation

print("Beam Search Generation\n")
print("=" * 60)

prompt = "Artificial intelligence is"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# Compare different beam sizes
beam_sizes = [1, 3, 5]

for num_beams in beam_sizes:
    print(f"\nBeam size: {num_beams}")
    print("-" * 60)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=50,
            num_beams=num_beams,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{generated_text}")

print("\n" + "=" * 60)
print("\nObservation: Larger beam sizes may produce different outputs")
print("but still deterministic for same beam size.")

In [ ]:
# Step 5: Beam Search with Multiple Outputs

print("Beam Search with Multiple Sequences\n")
print("=" * 60)

prompt = "The future of technology"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# Generate multiple sequences
print("Generating top 3 sequences with beam search:\n")
with torch.no_grad():
    outputs = model.generate(
        inputs['input_ids'],
        max_length=40,
        num_beams=5,
        num_return_sequences=3,  # Return top 3 sequences
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )

for i, output in enumerate(outputs, 1):
    text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"{i}. {text}")
    print()

print("=" * 60)
print("\nThese are the top 3 highest probability sequences found.")

## Part 3: Sampling - Adding Randomness

**Sampling** randomly selects the next token based on the probability distribution.

### How It Works:

```
At each step:
  1. Get probability distribution P
  2. Sample token ~ P (random choice weighted by probabilities)
  3. Add to sequence
```

### Advantages:
- Diverse outputs
- More creative and interesting
- Non-deterministic
- Can explore different possibilities

### Disadvantages:
- Can be incoherent
- May select low-probability tokens
- Less predictable quality
- Requires tuning

---

In [ ]:
# Step 6: Basic Sampling

print("Basic Sampling Generation\n")
print("=" * 60)

prompt = "Artificial intelligence is"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# Generate multiple times with sampling
print("Generating 5 times with sampling:\n")
for i in range(5):
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=50,
            do_sample=True,  # Enable sampling
            pad_token_id=tokenizer.eos_token_id
        )
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{i+1}. {text}")
    print()

print("=" * 60)
print("\nNotice: Each output is different (non-deterministic)")
print("Sampling introduces diversity but may be less coherent.")

## Part 4: Temperature - Controlling Randomness

**Temperature** controls the randomness of predictions by scaling the logits before softmax.

### How It Works:

```
logits = model_output / temperature
probabilities = softmax(logits)
```

### Temperature Values:

- **temperature = 1.0**: Original distribution (default)
- **temperature < 1.0**: More confident, peaked distribution (less random)
- **temperature > 1.0**: Flatter distribution (more random)
- **temperature → 0**: Approaches greedy search
- **temperature → ∞**: Approaches uniform distribution

### Effect on Distribution:

```
Original: [0.5, 0.3, 0.15, 0.05]

T = 0.5 (low):  [0.7, 0.2, 0.08, 0.02]  # More peaked
T = 1.0:        [0.5, 0.3, 0.15, 0.05]  # Original
T = 2.0 (high): [0.4, 0.3, 0.2,  0.1]   # Flatter
```

---

In [ ]:
# Step 7: Temperature Comparison

import matplotlib.pyplot as plt
import numpy as np

print("Temperature Effect on Generation\n")
print("=" * 60)

prompt = "The future of artificial intelligence"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# Test different temperatures
temperatures = [0.3, 0.7, 1.0, 1.5, 2.0]

for temp in temperatures:
    print(f"\nTemperature: {temp}")
    print("-" * 60)
    
    # Generate 2 samples for each temperature
    for i in range(2):
        with torch.no_grad():
            outputs = model.generate(
                inputs['input_ids'],
                max_length=40,
                do_sample=True,
                temperature=temp,
                pad_token_id=tokenizer.eos_token_id
            )
        
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"  {i+1}. {text}")
    print()

print("=" * 60)
print("\nObservations:")
print("  - Low temperature (0.3): More focused, repetitive, safe")
print("  - Medium temperature (1.0): Balanced creativity and coherence")
print("  - High temperature (2.0): More random, diverse, risky")

## Part 5: Top-k Sampling - Limiting Choices

**Top-k Sampling** restricts sampling to the k most likely tokens at each step.

### How It Works:

```
At each step:
  1. Get probability distribution
  2. Keep only top-k highest probabilities
  3. Renormalize probabilities
  4. Sample from top-k tokens
```

### Key Parameter:
- **top_k**: Number of highest probability tokens to consider (typically 40-50)

### Advantages:
- Prevents sampling very unlikely tokens
- More coherent than pure sampling
- Still allows diversity
- Simple to implement

### Disadvantages:
- Fixed k may not fit all contexts
- May cut off too many or too few tokens
- Doesn't adapt to distribution shape

---

In [ ]:
# Step 8: Top-k Sampling

print("Top-k Sampling Comparison\n")
print("=" * 60)

prompt = "Machine learning is"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# Test different top-k values
top_k_values = [10, 30, 50, 100]

for top_k in top_k_values:
    print(f"\nTop-k: {top_k}")
    print("-" * 60)
    
    # Generate 3 samples
    for i in range(3):
        with torch.no_grad():
            outputs = model.generate(
                inputs['input_ids'],
                max_length=40,
                do_sample=True,
                top_k=top_k,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id
            )
        
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"  {i+1}. {text}")
    print()

print("=" * 60)
print("\nObservations:")
print("  - Lower top-k (10): More focused, less diverse")
print("  - Higher top-k (100): More diverse, potentially less coherent")
print("  - Sweet spot often around 40-50")

## Part 6: Top-p (Nucleus) Sampling - Adaptive Selection

**Top-p (Nucleus) Sampling** selects from the smallest set of tokens whose cumulative probability exceeds p.

### How It Works:

```
At each step:
  1. Get probability distribution
  2. Sort tokens by probability (descending)
  3. Find smallest set where sum(probabilities) >= p
  4. Renormalize and sample from this set
```

### Example:

```
Tokens:     [A,   B,   C,   D,   E,   ...]
Probs:      [0.4, 0.3, 0.15, 0.1, 0.05, ...]
Cumulative: [0.4, 0.7, 0.85, 0.95, 1.0,  ...]

If p = 0.9:
  Nucleus = {A, B, C, D}  # Cumulative prob = 0.95 >= 0.9
  Sample from these 4 tokens only
```

### Key Parameter:
- **top_p**: Cumulative probability threshold (typically 0.9-0.95)

### Advantages:
- Adapts to distribution shape
- More tokens when distribution is flat
- Fewer tokens when distribution is peaked
- Generally better than top-k

### Disadvantages:
- Slightly more complex
- May still include unlikely tokens in flat distributions

---

In [ ]:
# Step 9: Top-p (Nucleus) Sampling

print("Top-p (Nucleus) Sampling Comparison\n")
print("=" * 60)

prompt = "Deep learning has revolutionized"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# Test different top-p values
top_p_values = [0.5, 0.7, 0.9, 0.95]

for top_p in top_p_values:
    print(f"\nTop-p: {top_p}")
    print("-" * 60)
    
    # Generate 3 samples
    for i in range(3):
        with torch.no_grad():
            outputs = model.generate(
                inputs['input_ids'],
                max_length=40,
                do_sample=True,
                top_p=top_p,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id
            )
        
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"  {i+1}. {text}")
    print()

print("=" * 60)
print("\nObservations:")
print("  - Lower top-p (0.5): More conservative, focused")
print("  - Higher top-p (0.95): More diverse, creative")
print("  - Common choice: 0.9 for good balance")

## Part 7: Combining Strategies

You can combine multiple strategies for optimal results.

### Common Combinations:

1. **Temperature + Top-p**
   - Most popular combination
   - Temperature controls randomness
   - Top-p prevents unlikely tokens

2. **Temperature + Top-k**
   - Alternative to top-p
   - Fixed vocabulary size

3. **Top-k + Top-p**
   - Double filtering
   - Very safe generation

---

In [ ]:
# Step 10: Combined Strategies

print("Combined Generation Strategies\n")
print("=" * 60)

prompt = "The impact of AI on society"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# Define strategy combinations
strategies = [
    {"name": "Greedy", "params": {"do_sample": False}},
    {"name": "Beam Search", "params": {"num_beams": 5, "do_sample": False}},
    {"name": "Pure Sampling", "params": {"do_sample": True}},
    {"name": "Temperature + Top-p", "params": {"do_sample": True, "temperature": 0.8, "top_p": 0.9}},
    {"name": "Temperature + Top-k", "params": {"do_sample": True, "temperature": 0.8, "top_k": 50}},
    {"name": "Top-k + Top-p", "params": {"do_sample": True, "top_k": 50, "top_p": 0.9}}
]

for strategy in strategies:
    print(f"\nStrategy: {strategy['name']}")
    print("-" * 60)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=50,
            pad_token_id=tokenizer.eos_token_id,
            **strategy['params']
        )
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{text}")
    print()

print("=" * 60)
print("\nEach strategy produces different characteristics!")

## Part 8: Visualizing Generation Strategies

Let's visualize how different strategies affect the probability distribution.

---

In [ ]:
# Step 11: Visualizing Strategy Effects

import matplotlib.pyplot as plt
import numpy as np

print("Visualizing Generation Strategies\n")
print("=" * 60)

# Simulate a probability distribution
np.random.seed(42)
logits = np.random.randn(20)
logits = np.sort(logits)[::-1]  # Sort descending

# Apply softmax to get probabilities
def softmax(x, temperature=1.0):
    x = x / temperature
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum()

# Original distribution
probs_original = softmax(logits, temperature=1.0)

# Different temperatures
probs_low_temp = softmax(logits, temperature=0.5)
probs_high_temp = softmax(logits, temperature=2.0)

# Top-k filtering (k=5)
probs_topk = probs_original.copy()
probs_topk[5:] = 0
probs_topk = probs_topk / probs_topk.sum()

# Top-p filtering (p=0.9)
probs_sorted = np.sort(probs_original)[::-1]
cumsum = np.cumsum(probs_sorted)
k_nucleus = np.argmax(cumsum >= 0.9) + 1
probs_topp = probs_original.copy()
threshold = probs_sorted[k_nucleus-1]
probs_topp[probs_topp < threshold] = 0
probs_topp = probs_topp / probs_topp.sum()

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Effect of Different Generation Strategies on Probability Distribution', 
             fontsize=16, fontweight='bold')

x = np.arange(20)

# Original
axes[0, 0].bar(x, probs_original, color='#4ECDC4', edgecolor='black', linewidth=1.5)
axes[0, 0].set_title('Original Distribution\n(Temperature = 1.0)', fontweight='bold')
axes[0, 0].set_ylabel('Probability', fontsize=11)
axes[0, 0].grid(axis='y', alpha=0.3)

# Low temperature
axes[0, 1].bar(x, probs_low_temp, color='#FF6B6B', edgecolor='black', linewidth=1.5)
axes[0, 1].set_title('Low Temperature\n(T = 0.5, More Peaked)', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# High temperature
axes[0, 2].bar(x, probs_high_temp, color='#95E1D3', edgecolor='black', linewidth=1.5)
axes[0, 2].set_title('High Temperature\n(T = 2.0, Flatter)', fontweight='bold')
axes[0, 2].grid(axis='y', alpha=0.3)

# Top-k
colors_topk = ['#FFD93D' if i < 5 else '#CCCCCC' for i in range(20)]
axes[1, 0].bar(x, probs_topk, color=colors_topk, edgecolor='black', linewidth=1.5)
axes[1, 0].set_title('Top-k Sampling\n(k = 5, Fixed Size)', fontweight='bold')
axes[1, 0].set_xlabel('Token Rank', fontsize=11)
axes[1, 0].set_ylabel('Probability', fontsize=11)
axes[1, 0].grid(axis='y', alpha=0.3)

# Top-p
colors_topp = ['#A8E6CF' if p > 0 else '#CCCCCC' for p in probs_topp]
axes[1, 1].bar(x, probs_topp, color=colors_topp, edgecolor='black', linewidth=1.5)
axes[1, 1].set_title(f'Top-p Sampling\n(p = 0.9, Nucleus Size = {k_nucleus})', fontweight='bold')
axes[1, 1].set_xlabel('Token Rank', fontsize=11)
axes[1, 1].grid(axis='y', alpha=0.3)

# Greedy (argmax)
probs_greedy = np.zeros(20)
probs_greedy[0] = 1.0
axes[1, 2].bar(x, probs_greedy, color='#C7CEEA', edgecolor='black', linewidth=1.5)
axes[1, 2].set_title('Greedy Search\n(Always Pick Highest)', fontweight='bold')
axes[1, 2].set_xlabel('Token Rank', fontsize=11)
axes[1, 2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization shows how each strategy modifies the probability distribution!")

## Part 9: Strategy Selection Guide

Choosing the right generation strategy depends on your use case.

---

In [ ]:
# Step 12: Strategy Comparison Table

import pandas as pd

print("Generation Strategy Comparison\n")
print("=" * 60)

# Create comparison table
data = {
    'Strategy': ['Greedy Search', 'Beam Search', 'Pure Sampling', 'Temperature', 'Top-k', 'Top-p (Nucleus)'],
    'Diversity': ['Very Low', 'Low', 'Very High', 'Adjustable', 'High', 'High'],
    'Quality': ['Medium', 'High', 'Variable', 'Adjustable', 'Good', 'Good'],
    'Speed': ['Fast', 'Slow', 'Fast', 'Fast', 'Fast', 'Fast'],
    'Deterministic': ['Yes', 'Yes', 'No', 'No', 'No', 'No'],
    'Best For': [
        'Translation, Summarization',
        'High-quality outputs',
        'Creative writing',
        'Controlling randomness',
        'Balanced generation',
        'Most tasks (recommended)'
    ]
}

df = pd.DataFrame(data)
print(df.to_string(index=False))
print("\n" + "=" * 60)

print("\n\nRecommended Settings by Use Case:\n")
print("-" * 60)

use_cases = [
    {
        'task': 'Translation',
        'strategy': 'Beam Search',
        'params': 'num_beams=5, early_stopping=True'
    },
    {
        'task': 'Summarization',
        'strategy': 'Beam Search',
        'params': 'num_beams=4, length_penalty=2.0'
    },
    {
        'task': 'Creative Writing',
        'strategy': 'Temperature + Top-p',
        'params': 'temperature=0.8, top_p=0.9'
    },
    {
        'task': 'Chatbot',
        'strategy': 'Temperature + Top-p',
        'params': 'temperature=0.7, top_p=0.9'
    },
    {
        'task': 'Code Generation',
        'strategy': 'Low Temperature + Top-p',
        'params': 'temperature=0.2, top_p=0.95'
    },
    {
        'task': 'Story Generation',
        'strategy': 'High Temperature + Top-p',
        'params': 'temperature=1.0, top_p=0.9'
    }
]

for uc in use_cases:
    print(f"\n{uc['task']}:")
    print(f"  Strategy: {uc['strategy']}")
    print(f"  Parameters: {uc['params']}")

print("\n" + "=" * 60)

## Part 10: Advanced Generation Parameters

Other important parameters for controlling generation.

---

In [ ]:
# Step 13: Advanced Generation Parameters

print("Advanced Generation Parameters\n")
print("=" * 60)

prompt = "In the year 2050"
print(f"Prompt: {prompt}\n")

inputs = tokenizer(prompt, return_tensors="pt")

# 1. Repetition Penalty
print("\n1. REPETITION PENALTY")
print("-" * 60)
print("Penalizes tokens that have already been generated.\n")

for penalty in [1.0, 1.5, 2.0]:
    print(f"Repetition penalty: {penalty}")
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=50,
            do_sample=True,
            temperature=0.8,
            repetition_penalty=penalty,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"  {text}")
    print()

# 2. Length Penalty (for beam search)
print("\n2. LENGTH PENALTY (Beam Search)")
print("-" * 60)
print("Controls preference for longer/shorter sequences.\n")

for length_penalty in [0.5, 1.0, 2.0]:
    print(f"Length penalty: {length_penalty}")
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=50,
            num_beams=5,
            length_penalty=length_penalty,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"  {text}")
    print()

# 3. No Repeat N-gram
print("\n3. NO REPEAT N-GRAM")
print("-" * 60)
print("Prevents repeating n-grams of specified size.\n")

for no_repeat_ngram_size in [0, 2, 3]:
    print(f"No repeat n-gram size: {no_repeat_ngram_size}")
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=50,
            do_sample=True,
            no_repeat_ngram_size=no_repeat_ngram_size,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"  {text}")
    print()

print("=" * 60)
print("\nThese parameters help control generation quality and diversity!")

## Exercises

### Exercise 1: Strategy Comparison
Compare all generation strategies on the same prompt:
```python
prompt = "The most important invention in history was"
# Test: greedy, beam search (3 sizes), sampling with different temperatures
# Analyze which produces the most interesting output
```

### Exercise 2: Temperature Exploration
Explore temperature effects systematically:
```python
temperatures = [0.1, 0.3, 0.5, 0.7, 0.9, 1.2, 1.5, 2.0]
# Generate 5 samples for each temperature
# Plot diversity vs coherence trade-off
```

### Exercise 3: Top-k vs Top-p
Compare top-k and top-p sampling:
```python
# Test top-k: [10, 20, 30, 40, 50]
# Test top-p: [0.5, 0.7, 0.8, 0.9, 0.95]
# Which produces better results for your use case?
```

### Exercise 4: Optimal Parameters
Find optimal parameters for different tasks:
- Creative story writing
- Technical documentation
- Conversational responses
- Code comments

Test combinations and document your findings.

### Exercise 5: Repetition Control
Experiment with repetition control:
```python
# Test repetition_penalty: [1.0, 1.2, 1.5, 2.0]
# Test no_repeat_ngram_size: [0, 2, 3, 4]
# Find settings that minimize repetition while maintaining coherence
```

---

## Key Takeaways

1. **Greedy search** always picks the highest probability token (fast but repetitive)
2. **Beam search** maintains multiple hypotheses for better quality (slower but more accurate)
3. **Sampling** introduces randomness for diverse outputs (creative but variable quality)
4. **Temperature** controls randomness: low = focused, high = diverse
5. **Top-k sampling** restricts choices to k most likely tokens (fixed size)
6. **Top-p (nucleus) sampling** adapts to distribution shape (recommended)
7. **Combining strategies** (temperature + top-p) often works best
8. **Repetition penalty** prevents repetitive text
9. **Length penalty** controls output length in beam search
10. **Strategy choice** depends on your specific use case

### Quick Reference:

**For Accuracy (Translation, Summarization):**
```python
generate(num_beams=5, early_stopping=True)
```

**For Creativity (Stories, Dialogue):**
```python
generate(do_sample=True, temperature=0.8, top_p=0.9)
```

**For Code/Technical:**
```python
generate(do_sample=True, temperature=0.2, top_p=0.95)
```

**For Balanced Output:**
```python
generate(do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.2)
```

---

## Additional Resources

### Papers
- **"The Curious Case of Neural Text Degeneration"** (Holtzman et al., 2019) - Introduces nucleus sampling
- **"Hierarchical Neural Story Generation"** (Fan et al., 2018) - Generation strategies for stories
- **"Get To The Point: Summarization with Pointer-Generator Networks"** (See et al., 2017) - Beam search for summarization

### Documentation
- [HuggingFace Generation Guide](https://huggingface.co/docs/transformers/main_classes/text_generation)
- [Generation Strategies](https://huggingface.co/blog/how-to-generate)
- [Decoding Methods](https://huggingface.co/docs/transformers/generation_strategies)

### Blog Posts
- [How to Generate Text with Transformers](https://huggingface.co/blog/how-to-generate)
- [Controlling Text Generation](https://towardsdatascience.com/controlling-text-generation-from-language-models)
- [Nucleus Sampling Explained](https://medium.com/@amanmehta_37887/nucleus-sampling-explained)

### Interactive
- [Text Generation Playground](https://huggingface.co/spaces) - Try different strategies
- [GPT-2 Demo](https://transformer.huggingface.co/doc/gpt2-large) - Interactive generation

---

## Next Lesson

**L11: Prompt Engineering 101** - Learn how to design effective prompts, use instruction formatting, create few-shot examples, and build prompt templates for better model outputs.

---

**Congratulations! You now understand text generation strategies!**

*You've learned how to control language model outputs using different decoding strategies and parameters.*